# PySpark with Delta Lake Example

This notebook demonstrates how to use PySpark locally in VS Code to work with Delta tables.

In [1]:
# Initialize PySpark Session with Delta Lake support
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Create Spark session
builder = SparkSession.builder \
    .appName("LocalDeltaLake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

AttributeError: module 'socketserver' has no attribute 'UnixStreamServer'

In [3]:
# Create sample data
data = [
    ("Alice", 34, "Engineering"),
    ("Bob", 45, "Sales"),
    ("Charlie", 28, "Marketing"),
    ("Diana", 52, "Engineering")
]

columns = ["Name", "Age", "Department"]

df = spark.createDataFrame(data, columns)
df.show()

NameError: name 'spark' is not defined

In [ ]:
# Write as Delta table to your Lakehouse
lakehouse_path = r"C:\Users\danie\OneLake - Microsoft\MSTR_db\dans_lake.Lakehouse\Tables"
table_name = "sample_employees"
table_path = f"{lakehouse_path}\\{table_name}"

# Write Delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(table_path)

print(f"Delta table written to: {table_path}")

NameError: name 'df' is not defined

In [ ]:
# Read Delta table back
df_read = spark.read.format("delta").load(table_path)
df_read.show()

print(f"Schema:")
df_read.printSchema()

In [ ]:
# Perform some PySpark operations
from pyspark.sql.functions import col, avg, count

# Group by department
dept_stats = df_read.groupBy("Department") \
    .agg(
        count("*").alias("Employee_Count"),
        avg("Age").alias("Average_Age")
    )

dept_stats.show()

In [ ]:
# Query using SQL
df_read.createOrReplaceTempView("employees")

result = spark.sql("""
    SELECT Department, COUNT(*) as count, AVG(Age) as avg_age
    FROM employees
    GROUP BY Department
    ORDER BY count DESC
""")

result.show()

In [ ]:
# Append new data
new_data = [
    ("Eve", 31, "Engineering"),
    ("Frank", 38, "Sales")
]

df_new = spark.createDataFrame(new_data, columns)

df_new.write \
    .format("delta") \
    .mode("append") \
    .save(table_path)

print("New data appended!")

# Read updated table
df_updated = spark.read.format("delta").load(table_path)
print(f"Total rows: {df_updated.count()}")
df_updated.show()

In [ ]:
# View Delta table history (time travel)
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, table_path)
history = delta_table.history()

print("Delta table history:")
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

In [ ]:
# Convert to Pandas for analysis
pandas_df = df_updated.toPandas()

print(type(pandas_df))
pandas_df.head()

In [ ]:
# Stop Spark session when done
spark.stop()
print("Spark session stopped")